In [17]:
print("测试成功")

测试成功


In [3]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms  
from PIL import Image
import os
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

#### class Mydataset

In [5]:

class MyDataset(Dataset):
    def __init__(self, root_dir, label_dir, transform=None):
        self.root_dir = root_dir
        self.label_dir = label_dir
        self.transform = transform

        self.path = os.path.join(root_dir, label_dir)
        self.image_files = [f for f in os.listdir(self.path) if f.endswith(('.jpg', '.png')) ]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_name = self.image_files[idx]
        img_item_path = os.path.join(self.path, img_name)
        image = Image.open(img_item_path).convert('RGB')
        label = self.label_dir

        if self.transform:
            image = self.transform(image)
        
        return image, label
    

In [ ]:
import os
print(os.getcwd())
print(os.path.exists("../datasets/train/ants"))
print(os.path.exists("../datasets/train/bees"))

d:\UserHE\LearningProject\pytorch\testfile
True
True


In [ ]:
root_dir = '../datasets/train'
ants_label_dir = 'ants'
bees_label_dir = 'bees'
ants_dataset = MyDataset(root_dir=root_dir, label_dir=ants_label_dir)
bees_dataset = MyDataset(root_dir=root_dir, label_dir=bees_label_dir)

In [ ]:
img, label = ants_dataset[0]    # 自动调用__getitem__方法
#img.show()

In [ ]:
import os
# save_labels_to_txt函数将图像文件夹中的每个图像文件的标签保存到一个单独的文本文件中。
# 文本文件的名称与图像文件相同，但扩展名为.txt，内容为标签。
def save_labels_to_txt(root_dir, source_dir):
    # root_dir = '../../datasets/train'
    # source_dir = 'ants_images'
    img_dir = os.path.join(root_dir, source_dir) 
    label = source_dir.split('_')[0]
    out_dir = os.path.join(root_dir, label + '_labels') 
    
    os.makedirs(out_dir, exist_ok=True)

    for file in [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]:
   
        file_name = os.path.splitext(file)[0]
        
        #print(f"Saving label for {file_name}: {label}")
        
        txt_path = os.path.join(out_dir, file_name + '.txt')
        with open(txt_path, 'w') as f:
            f.write(label)

In [ ]:
root_dir = '../datasets/train'

save_labels_to_txt(root_dir, 'ants_images')

Saving label for 0013035: ants
Saving label for 1030023514_aad5c608f9: ants
Saving label for 1095476100_3906d8afde: ants
Saving label for 1099452230_d1949d3250: ants
Saving label for 116570827_e9c126745d: ants
Saving label for 1225872729_6f0856588f: ants
Saving label for 1262877379_64fcada201: ants
Saving label for 1269756697_0bce92cdab: ants
Saving label for 1286984635_5119e80de1: ants
Saving label for 132478121_2a430adea2: ants
Saving label for 1360291657_dc248c5eea: ants
Saving label for 1368913450_e146e2fb6d: ants
Saving label for 1473187633_63ccaacea6: ants
Saving label for 148715752_302c84f5a4: ants
Saving label for 1489674356_09d48dde0a: ants
Saving label for 149244013_c529578289: ants
Saving label for 150801003_3390b73135: ants
Saving label for 150801171_cd86f17ed8: ants
Saving label for 154124431_65460430f2: ants
Saving label for 162603798_40b51f1654: ants
Saving label for 1660097129_384bf54490: ants
Saving label for 167890289_dd5ba923f3: ants
Saving label for 1693954099_46d4c

#### Tensorboard

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
# 每次记录时创建新的logs文件夹，或者删除之前的记录
writer = SummaryWriter(log_dir='runs/exp1')
img = ants_dataset[0][0]  
#img.show()
to_tensor_temp = transforms.ToTensor()
img_tensor = to_tensor_temp(img)

writer.add_image('Ant Image', img_tensor, 1)  
#writer.add_image(name, tensor, global_step=None, walltime=None, dataformats='CHW')
#tensor: (C, H, W) or (H, W, C) or (N, C, H, W) or (N, H, W, C)
#global_step: 全局步数，通常用于表示训练的迭代次数或批次编号. 
#global_step=None自动递增
#dataformats: 指定输入张量的格式，默认是'CHW'，表示通道、高度、宽度.
#numpy.array().shape: (H, W, C) 

for i in range(100):
    writer.add_scalar('Loss/train', 0.1 * i, i)


writer.close()

In [ ]:
#tensorboard --logdir=pytorch/notebook/runs/exp1 --port=6006

### Transforms

In [1]:
from torchvision import transforms

In [7]:
writer = SummaryWriter(log_dir='runs/exp2')

img = ants_dataset[10][0]

transform_1 = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
img_1 = transform_1(img)
writer.add_image('image1', img_1)

transform_2 = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor()
])
img_2 = transform_2(img)
writer.add_image('image2', img_2
)
writer.close()

In [ ]:
#tensorboard --logdir=pytorch/notebook/runs/exp2 --port=6006

### Train

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       #随机剪裁出32x32
    transforms.RandomHorizontalFlip(),          #随机水平翻转
    transforms.ToTensor(),      #图片像素（0-255）变成 PyTorch 能算的浮点数（0.0-1.0）
                                #并调整维度为 (通道, 高, 宽)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],    
                         std=[0.229, 0.224, 0.225])
                                #标准化。
                                #用这组均值和标准差将数据变成“均值为0，方差为1”的标准正态分布
                                #是 ImageNet 数据集的统计结果
])